# Inspect Recharge Target

QA/QC of the combined recharge calibration target written by
`targets/rch.py` to `<project>/targets/recharge_targets.nc` (and the
optional `recharge_targets_nn_filled.nc` companion).

The recharge target is a per-HRU per-year **`(lower_bound, upper_bound)`
range in dimensionless [0, 1]** (per `catalog/variables.yml` →
`recharge.units`). Up to three sources contribute on a shared fabric;
each is independently normalized to [0, 1] over `recharge.normalize_period`
and then combined with NaN-aware min / max:

- Reitz 2017 `total_recharge` (m/year → mm/year).
- WaterGAP 2.2d `qrdif` (kg m⁻² s⁻¹ monthly rate, summed to mm/year).
- ERA5-Land `ssro` (m/month, summed to mm/year).

The accompanying `n_sources` flag (0 / 1 / 2 / 3) records how many
sources contributed at each cell — the bound is NaN only when
`n_sources == 0`. The `nn_filled` companion file reuses the bounds
but additionally fills NaN HRUs from up to 10 nearest neighbours; the
`nn_filled` int8 flag (0 / 1) marks which HRU-years were filled.

**For the gfv2 fabric, `watergap22d` is excluded** (per the
`project_gfv2_coarse_grid_exclusions` memory: its 0.5° grid is too
coarse to resolve intermountain-west altitude gradients). The default
gfv2 target is therefore a 2-source bound (reitz + era5_land).

This notebook checks:

- File schema and global metadata (period, normalize_period,
  fabric SHA, source string).
- Per-time `n_sources` coverage and where the gaps fall geographically.
- `lower_bound`, `upper_bound`, and range size at a representative year.
- CONUS area-weighted mean lower / upper time series.
- Representative-HRU time series across four recharge regimes.
- NN-fill: which HRU-years were filled and whether the filled values
  preserve the lower / upper distribution.
- Bounds-in-[0,1] invariant check (period == normalize_period).

Companion to:

- `inspect_aggregated_recharge.ipynb` — per-source HRU aggregates
  before normalization + multi-source combination.

## Conventions

- HRU dim name follows `fabric.id_col` from the project config
  (e.g. `nat_hru_id` for the GFv2 fabric, `nhm_id` elsewhere).
- Bound units are dimensionless **`1`** (the normalize step strips
  physical units). Native source units are documented in
  `inspect_aggregated_recharge.ipynb`.
- `TARGET_YEAR` (set below) drives the at-time choropleth panels.
  Pick a year well inside the configured period — 2005 is a good
  default for the TM 6-B10 2000–2009 window.
- All maps are CONUS-Albers-equivalent only inside `area_weighted_mean`
  / `area_weighted_series` — the fabric is plotted in EPSG:4326 because
  the cost of reprojecting 360k polygons on every map is not worth it
  at inspection time.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    area_weighted_series,
    discover_target_nc,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    n_sources_per_time,
    nan_hru_count,
    open_target_nc,
    plot_categorical_choropleth,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/targets/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

TARGET = "recharge"
TARGET_YEAR = 2005  # any year in the configured period
# select_month also works for annual data when time is at year-start
# (the helper slices to a calendar-month window; year-start lands inside it).
TARGET_TIME = f"{TARGET_YEAR}-01-01"

# Four points spanning the wet / dry x cold / warm recharge regimes.
REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW conifer; high subsurface flow)": (-123.5, 47.8),
    "Iowa cropland (Midwest; recharge dominated by snowmelt + spring rain)": (-93.6, 42.0),
    "Phoenix metro (arid SW; near-zero recharge)": (-112.1, 33.4),
    "Southern Appalachians (Eastern broadleaf; high recharge)": (-83.5, 35.5),
}

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)
id_dim = fabric_cfg["id_col"]

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
fabric_path = fabric_cfg['path']
print(f"Fabric: {fabric_path} ({len(fabric)} HRUs, id_col={id_dim!r})")
print(f"Target year: {TARGET_YEAR}")

## Open and summarise the target NCs

`discover_target_nc` finds both the unfilled (`recharge_targets.nc`) and
NN-filled (`recharge_targets_nn_filled.nc`) variants if they exist.
Either may be `None` — this cell skips with a clear message and
downstream cells iterate only over what was loaded.

In [ ]:
raw_path, filled_path = discover_target_nc(project_dir, TARGET)

if raw_path is None:
    print(f"SKIP: {TARGET}_targets.nc not found at {project_dir / 'targets'}.")
    print("      Run `pixi run run-rch -- --project-dir <project>` first.")
    raise SystemExit

ds_raw = open_target_nc(raw_path)
print(f"Loaded raw target:    {raw_path.name}  ({raw_path.stat().st_size / 1e6:.1f} MB)")

ds_filled = None
if filled_path is not None:
    ds_filled = open_target_nc(filled_path)
    print(
        f"Loaded NN-filled:     {filled_path.name}  "
        f"({filled_path.stat().st_size / 1e6:.1f} MB)"
    )
else:
    print("NN-filled variant absent (set `nn_fill: true` in config to produce it).")

## Schema and global metadata

The global attrs carry the provenance needed at calibration time: the
output `period` and the `normalize_period` (the window over which each
source's min/max was computed), the fabric SHA-256, the comma-separated
`source` string (reflects only the active sources after any per-project
exclusions), and the references to TM 6-B10. Per-variable attrs carry
the units (`1` for the dimensionless bounds).

In [ ]:
print(ds_raw)
print()
print("=== Global attrs ===")
for k, v in ds_raw.attrs.items():
    print(f"  {k:<22} {v}")

print()
print("=== Per-variable units ===")
for v in ("lower_bound", "upper_bound", "n_sources"):
    units = ds_raw[v].attrs.get("units", "(no units attr)")
    long_name = ds_raw[v].attrs.get("long_name", "")
    print(f"  {v:<14} units={units!r}  long_name={long_name!r}")

## Per-time coverage

`n_sources_per_time` returns the per-year count of HRUs at each flag
value. The `n=0` column is the count of all-NaN HRUs at that year —
these are the cells the NN-fill targets. With gfv2's 2-source default
(reitz + era5_land), expect `n=2` to dominate; isolated `n=1` cells
indicate where one source's grid doesn't reach a fabric polygon.

In [ ]:
cov = n_sources_per_time(ds_raw)
print(cov.describe().T[["mean", "std", "min", "max"]])

fig, ax = plt.subplots(figsize=(11, 4))
cov.plot(ax=ax)
ax.set_xlabel("Time")
ax.set_ylabel("HRU count")
ax.set_title(f"Per-year n_sources distribution — {TARGET} target")
ax.legend(title="flag value", loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_coverage_timeseries")
plt.show()

## n_sources map at TARGET_YEAR

Categorical choropleth — colours match the `flag_values` attr on the
`n_sources` variable. NaN HRUs (where the year slice is missing) plot
in light grey; the legend reports the count of HRUs at each flag value
so coverage anomalies are visible at a glance.

In [ ]:
ns = select_month(ds_raw["n_sources"], TARGET_YEAR, 1).to_pandas()

categories = {
    0: ("0 sources (all NaN)", "crimson"),
    1: ("1 source", "khaki"),
    2: ("2 sources", "lightgreen"),
    3: ("3 sources", "darkgreen"),
}

fig, ax = plt.subplots(figsize=(11, 7))
plot_categorical_choropleth(
    ax,
    fabric,
    ns,
    categories=categories,
    title=f"n_sources at {TARGET_TIME} — {TARGET} target",
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_n_sources_map")
plt.show()

## Lower / upper / range maps at TARGET_YEAR

Three side-by-side panels of normalized [0, 1] values:

- `lower_bound` — NaN-aware min across the normalized sources.
- `upper_bound` — NaN-aware max across the normalized sources.
- `range = upper - lower` — the per-HRU per-year spread the NHM
  calibrator is asked to land inside. Wide ranges = sources disagree
  on whether this year was wet or dry relative to the calibration
  climatology; tight ranges = consensus among the sources at that cell.

Colour scale is fixed at [0, 1] for the bound panels (post-normalization)
and at [0, 1] for the range panel (worst-case range is 1.0).

In [ ]:
lb = select_month(ds_raw["lower_bound"], TARGET_YEAR, 1).to_pandas()
ub = select_month(ds_raw["upper_bound"], TARGET_YEAR, 1).to_pandas()
rng = ub - lb

units = ds_raw["lower_bound"].attrs.get("units", "1")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
plot_hru_choropleth(
    axes[0], fabric, lb, vmin=0, vmax=1, cmap="YlGnBu",
    title=f"lower_bound\n{TARGET_TIME} | {units}", units=units,
)
plot_hru_choropleth(
    axes[1], fabric, ub, vmin=0, vmax=1, cmap="YlGnBu",
    title=f"upper_bound\n{TARGET_TIME} | {units}", units=units,
)
plot_hru_choropleth(
    axes[2], fabric, rng, vmin=0, vmax=1, cmap="OrRd",
    title=f"range = upper - lower\n{TARGET_TIME} | {units}", units=units,
)
fig.suptitle(f"{TARGET} target bounds — {TARGET_TIME}", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_bounds_map")
plt.show()

print(f"lower CONUS area-weighted mean: {area_weighted_mean(lb, fabric):.4f} {units}")
print(f"upper CONUS area-weighted mean: {area_weighted_mean(ub, fabric):.4f} {units}")
print(f"range CONUS area-weighted mean: {area_weighted_mean(rng, fabric):.4f} {units}")

## NaN HRU coverage

HRUs where `n_sources == 0` (and therefore both bounds are NaN). These
are the cells that nearest-neighbour fill targets — at this year slice
they show as red. A handful of NaN HRUs is normal — typically tiny
fabric polygons that don't intersect any source's grid. A large red
blob means a source's spatial footprint isn't covering what we expect.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
plot_nan_hrus(
    ax, fabric, lb,
    title=(
        f"NaN HRUs (red) — {TARGET_TIME} | "
        f"{nan_hru_count(lb)} of {len(fabric)} "
        f"({100 * nan_hru_count(lb) / len(fabric):.2f}%)"
    ),
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_nan_map")
plt.show()

## CONUS area-weighted lower / upper time series

`area_weighted_series` reduces the (time, hru) bound arrays to a
per-year CONUS-mean, weighted by EPSG:5070 polygon area. Lower /
upper plotted together form an envelope — its width is the average
cross-source disagreement nationally.

When `normalize_period == period` (default), bounds are strictly in
[0, 1] by construction and the year-by-year envelope reflects the
interannual signal AS SEEN BY EACH SOURCE INDEPENDENTLY. Real drought
/ wet years should show up as the multi-source min lock-step (low) and
max lock-step (high).

In [ ]:
lb_series = area_weighted_series(ds_raw["lower_bound"], fabric, id_dim)
ub_series = area_weighted_series(ds_raw["upper_bound"], fabric, id_dim)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(
    lb_series.index, lb_series.values, ub_series.values,
    color="steelblue", alpha=0.25, label="lower–upper envelope",
)
ax.plot(lb_series.index, lb_series.values, color="steelblue", lw=1, label="lower")
ax.plot(ub_series.index, ub_series.values, color="darkblue", lw=1, label="upper")
ax.set_xlabel("Time")
ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
ax.set_title(f"{TARGET} target — CONUS area-weighted bounds")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_conus_series")
plt.show()

## Representative HRU time series

Four physiographically distinct points; `lookup_hrus_by_points` uses
`gpd.sjoin` to resolve each (lon, lat) to the containing HRU. The
lower / upper envelope at each point is the per-year spread the
calibrator targets locally — narrower envelopes = stronger inter-source
agreement at that cell.

In [ ]:
rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
print("Representative HRUs:", rep_hrus)

lb_at = ds_raw["lower_bound"].sel({id_dim: list(rep_hrus.values())})
ub_at = ds_raw["upper_bound"].sel({id_dim: list(rep_hrus.values())})

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
    lb_h = lb_at.sel({id_dim: hru_id}).values
    ub_h = ub_at.sel({id_dim: hru_id}).values
    t = pd.DatetimeIndex(lb_at.time.values)
    ax.fill_between(t, lb_h, ub_h, color="steelblue", alpha=0.25)
    ax.plot(t, lb_h, color="steelblue", lw=1, label="lower")
    ax.plot(t, ub_h, color="darkblue", lw=1, label="upper")
    ax.set_title(f"{label} (HRU {hru_id})")
    ax.set_ylabel(f"recharge ({units})")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
fig.suptitle(f"{TARGET} target at representative HRUs", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_representative_series")
plt.show()

## NN-fill comparison

The companion `recharge_targets_nn_filled.nc` reuses the same bounds but
fills NaN HRUs from up to `nn_max_candidates=10` nearest neighbours
(see `targets/rch.py`). The `nn_filled` int8 flag marks each
HRU-year as `0` (not filled) or `1` (filled).

Two checks:

- **Filled-flag map at TARGET_YEAR** — should match the NaN-HRU map
  one-for-one (any NaN that *can* be filled gets a `1`).
- **Distribution preservation** — the area-weighted CONUS mean of the
  filled bounds should track the unfilled mean closely (the NN fill
  is interpolating from immediate neighbours, not synthesising). A
  large jump indicates an over-aggressive fill.

In [ ]:
if ds_filled is None:
    print("SKIP: NN-filled variant not present.")
else:
    flag = select_month(ds_filled["nn_filled"], TARGET_YEAR, 1).to_pandas()
    n_filled = int((flag == 1).sum())
    print(
        f"NN-filled at {TARGET_TIME}: {n_filled} HRUs filled "
        f"({100 * n_filled / len(fabric):.2f}%)"
    )

    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    plot_categorical_choropleth(
        axes[0], fabric, flag,
        categories={0: ("not filled", "lightgrey"), 1: ("filled", "crimson")},
        title=f"nn_filled flag at {TARGET_TIME}",
    )
    lb_f = select_month(ds_filled["lower_bound"], TARGET_YEAR, 1).to_pandas()
    diff = (lb_f - lb).abs().fillna(lb_f)
    plot_hru_choropleth(
        axes[1], fabric, diff, vmin=0, vmax=0.25, cmap="OrRd",
        title=f"|lower_bound_filled - lower_bound_raw| at {TARGET_TIME}",
        units=units,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_map")
    plt.show()

    lb_f_series = area_weighted_series(ds_filled["lower_bound"], fabric, id_dim)
    ub_f_series = area_weighted_series(ds_filled["upper_bound"], fabric, id_dim)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(lb_series.index, lb_series.values, color="steelblue", lw=1, label="lower (raw)")
    ax.plot(lb_f_series.index, lb_f_series.values, color="steelblue", lw=1, ls="--", label="lower (filled)")
    ax.plot(ub_series.index, ub_series.values, color="darkblue", lw=1, label="upper (raw)")
    ax.plot(ub_f_series.index, ub_f_series.values, color="darkblue", lw=1, ls="--", label="upper (filled)")
    ax.set_xlabel("Time")
    ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
    ax.set_title(f"{TARGET} target — raw vs NN-filled CONUS series")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_series")
    plt.show()

## Bounds-in-[0, 1] invariant check

When `normalize_period == period` (the default), every finite value of
both bounds must be in [0, 1] by construction (each source is min-max
normalized over the same window the output covers, so the per-source
[0, 1] cap propagates through the min/max combiner).

When `normalize_period != period` (extended-output configurations),
values outside the normalize window MAY fall outside [0, 1] — that's
documented as "visibly out-of-range" behavior, not a bug. The check
below distinguishes the two cases.

In [ ]:
lb_arr = ds_raw["lower_bound"].values
ub_arr = ds_raw["upper_bound"].values
finite_pair = np.isfinite(lb_arr) & np.isfinite(ub_arr)

lb_min = float(np.nanmin(lb_arr))
lb_max = float(np.nanmax(lb_arr))
ub_min = float(np.nanmin(ub_arr))
ub_max = float(np.nanmax(ub_arr))
lower_le_upper = bool((ub_arr[finite_pair] >= lb_arr[finite_pair]).all())

print(f"lower_bound range: [{lb_min:.4f}, {lb_max:.4f}]")
print(f"upper_bound range: [{ub_min:.4f}, {ub_max:.4f}]")
print(f"upper >= lower at every finite pair: {lower_le_upper}")

period = ds_raw.attrs.get("period", "")
norm_period = ds_raw.attrs.get("normalize_period", "")
if period == norm_period:
    print(
        f"period == normalize_period ({period!r}); bounds MUST be in [0, 1]."
    )
    assert lb_min >= -1e-6 and ub_max <= 1 + 1e-6, "Bound out-of-range"
    assert lower_le_upper, "upper < lower somewhere"
    print("PASS: bounds in [0, 1] and upper >= lower everywhere.")
else:
    print(
        f"period {period!r} != normalize_period {norm_period!r} — bounds outside "
        "[0, 1] are expected at extended timesteps. No assertion."
    )

In [ ]:
ds_raw.close()
if ds_filled is not None:
    ds_filled.close()